# FireSight Stage B: E0 vs E1 (prepared, not launched)

Fresh `yolo11s.pt`, 20 epochs, patience 10, checkpoint selection by **recall at FPR<=2% on val_cal**.
E0 = 25% list as-is. E1 = E0 + 1,327 added hard/easy negatives (9.0% of the 14,747-image list).
No GPU run until the lead approves. TEST/SEALED and `Sample_videos` are never used.

In [ ]:
import subprocess, os, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "GPU required"
print("gpus", torch.cuda.device_count())

In [ ]:
import importlib.metadata as md, os
os.environ["YOLO_AUTOINSTALL"]="False"; os.environ["YOLO_VERBOSE"]="False"; os.environ["YOLO_OFFLINE"]="True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
def ver(n):
    try: return md.version(n)
    except Exception: return None
if ver("ultralytics") != "8.4.147":
    get_ipython().system("pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop")
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")})

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent
CODE = sorted(Path("/kaggle/input").rglob("stage_b_train.py"))[0].parent
NEG = sorted(Path("/kaggle/input").rglob("e1_negatives.txt"))[0].parent.parent
EVALNEG = sorted(Path("/kaggle/input").rglob("eval_neg.txt"))[0].parent.parent
YOLO = sorted(Path(CODE).glob("yolo11s.pt"))[0]
print("DATA", DATA); print("CODE", CODE); print("NEG", NEG); print("EVALNEG", EVALNEG); print("YOLO", YOLO)

In [ ]:
from pathlib import Path
def resolve(line, root):
    p = Path(line.strip())
    return p if p.is_absolute() else (Path(root)/p).resolve()
out = Path("/kaggle/working/lists"); out.mkdir(parents=True, exist_ok=True)
base = [resolve(l, DATA) for l in open(DATA/"lists/train_25.txt", encoding="utf-8") if l.strip()]
neg  = [resolve(l, NEG) for l in open(NEG/"lists/e1_negatives.txt", encoding="utf-8") if l.strip()]
val  = [resolve(l, DATA) for l in open(DATA/"lists/val_early.txt", encoding="utf-8") if l.strip()]
cal  = [resolve(l, DATA) for l in open(DATA/"lists/val_cal.txt", encoding="utf-8") if l.strip()]
rep  = [resolve(l, DATA) for l in open(DATA/"lists/val_report.txt", encoding="utf-8") if l.strip()]
indep= [resolve(l, EVALNEG) for l in open(EVALNEG/"lists/eval_neg.txt", encoding="utf-8") if l.strip()]
for name, items in [("e0_train.txt",base),("e1_train.txt",base+neg),("val.txt",val),("val_cal.txt",cal),("val_report.txt",rep),("eval_neg.txt",indep)]:
    (out/name).write_text("\n".join(str(p) for p in items)+"\n", encoding="utf-8")
    print(name, len(items))
assert len(base)==14747 and len(neg)==1327 and len(indep)==5301

In [ ]:
import subprocess, json, sys
def run(arm, train_list):
    cmd = ["python","-u",str(Path(CODE)/"stage_b_train.py"),
           "--dataset",str(DATA),"--train-list",str(Path("/kaggle/working/lists")/train_list),
           "--val-list",str(Path("/kaggle/working/lists/val.txt")),
           "--val-cal-list",str(Path("/kaggle/working/lists/val_cal.txt")),
           "--out",f"/kaggle/working/runs/{arm}","--model",str(YOLO),
           "--epochs","20","--patience","10","--imgsz","640","--batch","64",
           "--device","0","--workers","4","--lr0","0.01","--lrf","0.01","--seed","0","--amp"]
    print("RUN", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
run("E0","e0_train.txt")

In [ ]:
run("E1","e1_train.txt")

In [ ]:
for arm in ("E0","E1"):
    preds=f"/kaggle/working/preds_{arm}.jsonl"
    subprocess.run(["python",str(Path(CODE)/"stage_b_predict.py"),"--model",f"/kaggle/working/runs/{arm}/weights/best_recall.pt",
                    "--list","/kaggle/working/lists/val_report.txt","--out",preds,"--device","0","--half"],check=True)
    for t in ("0.009","0.015","0.02","0.03"):
        subprocess.run(["python",str(Path(CODE)/"pilot_eval.py"),"--dataset",str(DATA),"--preds",preds,
                        "--model",arm,"--target-fpr",t,"--out",f"/kaggle/working/metrics_{arm}_{t}.json"],check=True)

In [ ]:
for arm in ("E0","E1"):
    subprocess.run(["python",str(Path(CODE)/"stage_b_predict.py"),"--model",f"/kaggle/working/runs/{arm}/weights/best_recall.pt",
                    "--list","/kaggle/working/lists/eval_neg.txt","--out",f"/kaggle/working/preds_{arm}_indep.jsonl",
                    "--device","0","--half"],check=True)
print("Independent-negative predictions written. Gate 1 comparison: docs/GATE_RESULTS.md")

In [ ]:
for arm in ("E0","E1"):
    print("\n==",arm,"==")
    print(open(f"/kaggle/working/runs/{arm}/run_summary.json").read())
    for t in ("0.009","0.015","0.02","0.03"):
        m=json.load(open(f"/kaggle/working/metrics_{arm}_{t}.json"))
        c=m["splits"]["val"]["combined"]; f=m["splits"]["val"]["fpr_per_negative_image"]
        print(t, "thr",m["thresholds"],"recall",round(c["recall"],4),"fpr",round(f["fpr"],4))